In [8]:
%%capture
from pathlib import Path

if Path.cwd().stem == "notebooks":
    %cd ..
    %load_ext autoreload
    %autoreload 2

In [9]:
import os
from pathlib import Path

import numpy as np
import polars as pl
import tomllib
from dotenv import load_dotenv
from polars import col

from notebooks.figures import (
    calculate_z_score,
    compose_panel_figure,
    plot_stimulus_seed_grid,
    plot_stimulus_with_labels,
    plot_stimulus_with_physiological_signals,
)
from src.data.database_manager import DatabaseManager
from src.data.stimulus_generator import StimulusGenerator
from src.features.scaling import scale_min_max
from src.features.utils import to_describe
from src.log_config import configure_logging

configure_logging(
    ignore_libs=("Comm", "bokeh", "tornado", "matplotlib"),
)

pl.Config.set_tbl_rows(12)  # for 12 seeds

polars.config.Config

In [10]:
db = DatabaseManager()

with db:
    df = db.get_trials("Explore_Data", exclude_problematic=True)


In [11]:
df = df.rename({"rating": "pain_rating", "pupil": "pupil_diameter"})
df = scale_min_max(
    df,
    exclude_additional_columns=[
        # already normalized:
        "temperature",
        "pain_rating",
        "brow_furrow",
        "cheek_raise",
        "mouth_open",
        "upper_lip_raise",
        "nose_wrinkle",
    ],
)

In [12]:
width = 800
height = 300

## Curve with physiological signals and confidence intervals

In [13]:
signals = [
    "temperature",
    "pain_rating",
    "pupil_diameter",
    "eda_tonic",
    "eda_phasic",
    "heart_rate",
    "mouth_open",
]

confidence_level = 0.95
z_score = calculate_z_score(confidence_level)

df = df.filter(col("stimulus_seed") == 133)

# Group by stimulus seed and normalized timestamp, then calculate mean, std, sem, ci
ci_values = (
    df.group_by(col("normalized_timestamp"), maintain_order=True)
    .agg(
        *[col(c).mean().alias(f"mean_{c}") for c in signals],
        *[col(c).std().alias(f"std_{c}") for c in signals],
        pl.len().alias("n"),
    )
    .sort("normalized_timestamp")
    .with_columns(
        *[(col(f"std_{c}") / col("n").sqrt()).alias(f"sem_{c}") for c in signals],
    )
    .with_columns(
        *[
            (col(f"mean_{c}") - z_score * col(f"sem_{c}")).alias(f"ci_lower_{c}")
            for c in signals
        ],
        *[
            (col(f"mean_{c}") + z_score * col(f"sem_{c}")).alias(f"ci_upper_{c}")
            for c in signals
        ],
    )
)

In [14]:
phy = plot_stimulus_with_physiological_signals(
    ci_values,
    signals,
    width=width,
    height=height,
)
phy

alt.LayerChart(...)

## Curve with temperature intervals

In [15]:
def load_configuration(file_path: str) -> dict:
    """Load configuration from a TOML file."""
    file_path = Path(file_path)
    with open(file_path, "rb") as file:
        return tomllib.load(file)


config = load_configuration("src/data/stimulus_config.toml")["stimulus"]

dummy_participant = {
    "temperature_baseline": 44.5,
    "temperature_range": 3,  # VAS 0 - VAS 70
}
config.update(dummy_participant)

stimulus = StimulusGenerator(config, seed=133)


In [16]:
lab = plot_stimulus_with_labels(
    stimulus,
    width=width,
    height=height,
)

lab

alt.LayerChart(...)

## All curves 


In [17]:
config["sample_rate"] = 2

stimuli = pl.concat(
    [
        pl.DataFrame(
            {
                "y": StimulusGenerator(config, seed).y,
                "time": np.arange(len(StimulusGenerator(config, seed).y)),
                "seed": np.array([seed] * len(StimulusGenerator(config, seed).y)),
            }
        )
        for seed in config["seeds"]
    ]
)
stimuli.group_by("seed", maintain_order=True).agg(to_describe("y"))

seed,y_count,y_null_count,y_mean,y_std,y_min,y_25%,y_50%,y_75%,y_max
i64,u32,u32,f64,f64,f64,f64,f64,f64,f64
133,360,0,44.338389,0.858954,43.0,43.734962,44.091604,44.980148,45.993009
243,360,0,44.376774,0.847801,43.0,43.750825,44.386525,45.055987,45.977269
265,360,0,44.394133,0.887838,43.0,43.679149,44.388702,45.081144,45.988746
396,360,0,44.466707,0.830799,43.0,43.952242,44.410173,45.118792,45.963669
467,360,0,44.425745,0.850121,43.0,43.860424,44.371972,45.087184,45.974603
658,360,0,44.425022,0.835784,43.0,43.727221,44.433312,45.03424,45.909149
681,360,0,44.3455,0.829624,43.0,43.721686,44.27416,44.990047,45.979076
743,360,0,44.434506,0.830132,43.0,43.916819,44.374246,45.08161,45.977096
806,360,0,44.320889,0.893898,43.0,43.557677,44.318286,44.99707,45.998544


In [18]:
chart = plot_stimulus_seed_grid(
    stimuli, columns=3, width=300, height=100, font_size=18, title_x_offset=20
)
chart = chart.configure_header(labelFontSize=2)

chart

alt.ConcatChart(...)

In [28]:
load_dotenv()
FIGURE_DIR = Path(os.getenv("SCI_DATA_FIGURE_DIR"))

phy.save(FIGURE_DIR / "stimulus_with_physiological_signals_ci.svg")
lab.save(FIGURE_DIR / "stimulus_with_labels_ci.svg")
chart.save(FIGURE_DIR / "stimulus_seed_grid.svg")

In [30]:
%%capture
compose_panel_figure(
    output_path=FIGURE_DIR / "panel_figure.svg",
    row1=FIGURE_DIR / "stimulus_with_labels_ci.svg",
    row2_left=FIGURE_DIR / "stimulus_seed_grid.svg",
    row2_right=FIGURE_DIR / "forearm_sci_dat.png",
)


17:03:01 | DEBUG   | PIL.PngImagePlugin | STREAM b'IHDR' 16 13
17:03:01 | DEBUG   | PIL.PngImagePlugin | STREAM b'pHYs' 41 9
17:03:01 | DEBUG   | PIL.PngImagePlugin | STREAM b'IDAT' 62 8192
